# PhysSkin RigNet visualization

This notebook loads a pretrained checkpoint and a processed RigNet sample, predicts its neural skinning fields and visualizes every field. Run the cells in order from the repository root.

## 1. Imports and paths

In [4]:
import gc
import os
from pathlib import Path

import ipywidgets as widgets
import k3d
import numpy as np
import torch
from IPython.display import display

from physskin.checkpoint import load_model, predict_weights
from physskin.skinning import standard_lbs

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Use the processed examples shipped with this repository by default.
EXAMPLE_MODEL_IDS = ['10559', '15668', '17959', '2132', '17872', '1996', '731', '761', '7199', '3220']
DATASET_ROOT = Path(os.environ.get('DATASET_ROOT', 'data/examples'))
MODEL_ID = os.environ.get('MODEL_ID', '10559')
CHECKPOINT_PATH = Path('checkpoints/physskin_rignet_epoch150.pt')

# 0 keeps all interior points. Set a positive
# value (for example 30000) if interactive rendering is slow on your machine.
MAX_POINTS = int(os.environ.get('MAX_POINTS', '0'))

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'RigNet dataset not found at {DATASET_ROOT}. Edit DATASET_ROOT in this cell.'
    )
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH.resolve()}')

print('device:', device)
print('dataset:', DATASET_ROOT.resolve())
print('model id:', MODEL_ID)
print('checkpoint:', CHECKPOINT_PATH.resolve())


device: cuda
dataset: /mnt/nas_9/group/leiyuanhang/zju3dv_codes/PhysSkin/data/examples
model id: 10559
checkpoint: /mnt/nas_9/group/leiyuanhang/zju3dv_codes/PhysSkin/checkpoints/physskin_rignet_epoch150.pt


## 2. Load the checkpoint and predict skinning fields

In [5]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model, config = load_model(CHECKPOINT_PATH, device=device)
sample_dir = DATASET_ROOT / '00000001' / MODEL_ID / 'models' / 'samples'
latent_file = sample_dir / 'latents.npz'
points_file = sample_dir / 'internal_filled.npz'
if not latent_file.is_file() or not points_file.is_file():
    raise FileNotFoundError(
        f'Expected latents.npz and internal_filled.npz under {sample_dir}'
    )
latents_np = np.load(latent_file)['latents']
points_np = np.load(points_file)['points'] * 2.0  # [-0.5, 0.5] -> [-1, 1]
latents = torch.tensor(latents_np, dtype=torch.float32, device=device)
points = torch.tensor(points_np, dtype=torch.float32, device=device)

if MAX_POINTS > 0 and points.shape[0] > MAX_POINTS:
    generator = torch.Generator(device=points.device).manual_seed(0)
    indices = torch.randperm(points.shape[0], generator=generator, device=points.device)[:MAX_POINTS]
    points = points[indices]

# Evaluate all selected points together to apply global L2 normalization.
weights = predict_weights(model, latents, points)

print('latents:', tuple(latents.shape))
print('points:', tuple(points.shape))
print('weights:', tuple(weights.shape))
print('learned handles:', config['num_handles'])
print('last channel: fixed rigid mode')

for mode_index in range(weights.shape[1]):
    values = weights[:, mode_index]
    print(
        f'mode {mode_index:02d}: min={values.min().item(): .6f}, '
        f'max={values.max().item(): .6f}, mean={values.mean().item(): .6f}, '
        f'l2={torch.linalg.vector_norm(values).item(): .6f}'
    )


latents: (1, 256, 768)
points: (121462, 3)
weights: (121462, 17)
learned handles: 16
last channel: fixed rigid mode
mode 00: min=-0.000293, max= 0.018123, mean= 0.000655, l2= 1.000000
mode 01: min=-0.013220, max= 0.000367, mean=-0.000929, l2= 1.000000
mode 02: min=-0.019862, max= 0.000466, mean=-0.000667, l2= 1.000000
mode 03: min=-0.000338, max= 0.019879, mean= 0.000510, l2= 1.000000
mode 04: min=-0.000412, max= 0.019998, mean= 0.000494, l2= 1.000000
mode 05: min=-0.018100, max= 0.000375, mean=-0.000602, l2= 1.000000
mode 06: min=-0.016691, max= 0.000511, mean=-0.000855, l2= 1.000000
mode 07: min=-0.017222, max= 0.000320, mean=-0.000623, l2= 1.000000
mode 08: min=-0.000632, max= 0.021424, mean= 0.000700, l2= 1.000000
mode 09: min=-0.000367, max= 0.014700, mean= 0.000928, l2= 1.000000
mode 10: min=-0.001013, max= 0.021900, mean= 0.000590, l2= 1.000000
mode 11: min=-0.020515, max= 0.000379, mean=-0.000548, l2= 1.000000
mode 12: min=-0.021438, max= 0.000430, mean=-0.000514, l2= 1.000000


## 3. Interactive skinning-field viewer

Use the mode slider to inspect all 16 learned fields and the final rigid mode. Positive values are red, negative values are blue, and values near zero are white.

In [6]:
show_points = points.detach().cpu().numpy()
show_weights = weights.detach().cpu().numpy()
num_modes = show_weights.shape[1]

def rgb_to_uint32(rgb):
    rgb = np.asarray(rgb, dtype=np.uint32)
    return (rgb[:, 0] << 16) + (rgb[:, 1] << 8) + rgb[:, 2]

def mode_colors(mode_index, signed=True, clip_percentile=99.0):
    values = np.asarray(show_weights[:, mode_index], dtype=np.float32)
    if signed:
        scale = np.percentile(np.abs(values), clip_percentile)
        scale = 1.0 if scale < 1e-12 else scale
        normalized = np.clip(values / scale, -1.0, 1.0)
        rgb = np.ones((values.shape[0], 3), dtype=np.float32)
        negative = normalized < 0
        rgb[negative, 0] = 1.0 + normalized[negative]
        rgb[negative, 1] = 1.0 + normalized[negative]
        positive = ~negative
        rgb[positive, 1] = 1.0 - normalized[positive]
        rgb[positive, 2] = 1.0 - normalized[positive]
    else:
        low = np.percentile(values, 100.0 - clip_percentile)
        high = np.percentile(values, clip_percentile)
        high = low + 1.0 if abs(high - low) < 1e-12 else high
        normalized = np.clip((values - low) / (high - low), 0.0, 1.0)
        rgb = np.stack([
            normalized,
            0.25 + 0.5 * (1.0 - np.abs(normalized - 0.5) * 2.0),
            1.0 - normalized,
        ], axis=1)
    return rgb_to_uint32(np.clip(rgb * 255.0, 0, 255).astype(np.uint32))

mode_slider = widgets.IntSlider(
    value=0, min=0, max=num_modes - 1, step=1, description='mode'
)
signed_toggle = widgets.Checkbox(value=True, description='signed colors')
clip_slider = widgets.FloatSlider(
    value=99.0, min=90.0, max=100.0, step=0.1, description='clip %'
)
size_slider = widgets.FloatLogSlider(
    value=0.01, base=10, min=-3.0, max=-0.5, step=0.05, description='pt size'
)
mode_info = widgets.HTML()

weight_plot = k3d.plot(grid_visible=True, height=720)
weight_points = k3d.points(
    show_points,
    colors=mode_colors(0),
    point_size=0.01,
    shader='flat',
)
weight_plot += weight_points

def update_weight_view(*_):
    mode_index = int(mode_slider.value)
    values = show_weights[:, mode_index]
    weight_points.colors = mode_colors(
        mode_index,
        signed=bool(signed_toggle.value),
        clip_percentile=float(clip_slider.value),
    )
    weight_points.point_size = float(size_slider.value)
    mode_info.value = (
        f'<b>mode {mode_index}</b> &nbsp; min={values.min():.6g}, '
        f'max={values.max():.6g}, mean={values.mean():.6g}, '
        f'l2={np.linalg.norm(values):.6g}'
    )

for widget in (mode_slider, signed_toggle, clip_slider, size_slider):
    widget.observe(update_weight_view, names='value')

update_weight_view()
display(widgets.VBox([
    widgets.HBox([mode_slider, signed_toggle, clip_slider, size_slider]),
    mode_info,
    weight_plot,
]))
